<!-- cabecera-entorno -->
## Antes de empezar

**Clase 9 · Data storytelling** — Bloque 2 · Demo. Este cuaderno se recorre **por su cuenta**:
explica cada concepto antes de usarlo, y el profesor circula por el salón resolviendo dudas. No hay
que esperar a que alguien lo dicte desde el tablero.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `demo.ipynb` como
`demo_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/EJECUCION_PRESUPUESTAL.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 9 · Demo — De la cifra a la historia

**Caso:** la brecha entre lo que el Estado colombiano se comprometió a gastar y lo que efectivamente
pagó.

**Dataset:** `../datos/EJECUCION_PRESUPUESTAL.csv` — ejecución del Presupuesto General de la
Nación, vía datos.gov.co. 3.645 filas x 21 columnas. Una fila = un rubro presupuestal de una entidad.

## Cómo se usa este cuaderno

Está escrito para que usted avance **solo**. Cada bloque de código viene precedido de la explicación
del concepto que usa, y cada término nuevo se define la primera vez que aparece. **Hay que leer antes
de ejecutar.**

Este notebook tiene dos mitades y son muy distintas:

| Mitad | Secciones | Qué se hace | Herramienta |
|-------|-----------|-------------|-------------|
| Primera | 1 a 5 | Calcular las cifras del caso | pandas |
| Segunda | 6 a 9 | Escribir la historia de 4 slides | celdas markdown |

**La segunda mitad no tiene código y eso es a propósito.** El objeto de aprendizaje de hoy es la
decisión narrativa, no la sintaxis. Un notebook que empieza siendo herramienta de cálculo y termina
siendo cuaderno de redacción es exactamente lo que pasa en un trabajo real.

**El recorrido:**

| Sección | De qué va | Qué se lleva |
|---------|-----------|--------------|
| 1 | El dataset, el glosario presupuestal y los dos nombres sucios | Saber qué significa cada columna antes de sumarla |
| 2 | Las cuatro cifras del país, y por qué se dividen entre un billón | Una cifra legible |
| 3 | La brecha y el porcentaje de ejecución | El titular |
| 4 | Dónde está la brecha: `groupby`, `sort_values`, `head` | El nombre propio de la historia |
| 5 | El recorte: la misma brecha partida por tipo de gasto | Que una cifra sin su recorte no significa nada |
| 6 | Las cuatro slides: contexto, hallazgo, implicación, acción | El marco completo, escrito |
| 7 | La prueba de los títulos | Cómo saber en 20 segundos si el deck está mal |
| 8 | La misma verdad para tres audiencias | Elegir una audiencia y sostenerla |
| 9 | Preguntas que siempre salen, y el resumen | Las reglas que se llevan al reto |

**Aquí no hay nada que teclear.** Todo el código está escrito y ejecutable: usted lo corre, mira la
salida y lee la explicación que está justo encima. **Escribir código es el bloque 3**, con el reto, y
es lo que se entrega. Tampoco hay verificador: este cuaderno no tiene celdas que le digan CORRECTO,
porque hoy no hay nada que acertar.

**Las doce preguntas de interpretación** son lo que sí le toca a usted. No llevan código: se
responden escribiendo en español en la celda, debajo de *Tu respuesta:*. Cada una trae un bloque
plegable *"Comparar con la respuesta esperada"*. **Escriba la suya primero y ábralo después.**
Abrirlo antes no le ahorra nada: lo que se evalúa en el reto y en la sustentación del Momento 2 es
que usted sepa mirar un resultado y decir qué significa, no que sepa reproducir un cálculo.

**Las cajas "Para entender qué está pasando"** explican el fundamento: por qué la cifra se divide,
por qué el recorte cambia la conclusión, dónde está la línea entre analítica y opinión. Si ya las
tiene claras, se pueden saltar sin perder el hilo; si no, son la parte que más va a servirle en la
sustentación del Momento 2.

---

### Lo que este cuaderno da por sabido

No se repite aquí nada de lo que ya se explicó. Si algo de esta lista no le suena, vuelva al cuaderno
que lo enseña **antes** de seguir.

| Lo que se da por sabido | Dónde se explicó |
|-------------------------|------------------|
| Librería, alias, DataFrame, Series, índice, dtype | Clase 2, demo, secciones 1 a 5 |
| `read_csv`, `head`, `shape`, seleccionar columnas con `df[[...]]` | Clase 2, demo, secciones 2 y 3 |
| Máscara booleana para filtrar filas | Clase 2, demo, secciones 7 a 11 |
| `groupby` y la analogía de los M&Ms | Clase 4, demo, sección 4 |
| `sort_values` y leer una tabla ordenada | Clase 4, demo, sección 4 |
| Qué gráfico corresponde a cada par de tipos | Clase 5, demo, sección 1 |
| Data-ink ratio: quitar todo lo que no comunica | Clase 8, demo, completa |

**Lo nuevo de hoy** no es ninguna función de pandas. Es la decisión de **qué cifra se cuenta, en qué
orden y para quién**. Por eso el código de este cuaderno es todo conocido: si hoy hubiera sintaxis
nueva, la sintaxis se llevaría la atención.

---

## 1. El dataset, antes de tocarlo

**El concepto: un dataset presupuestal no se entiende leyendo los nombres de las columnas.** Cuatro
de las 21 columnas son plata, y las cuatro suenan parecido. Sumarlas sin saber qué significan produce
frases que suenan bien y son falsas.

**El glosario, primero. Sin esto las columnas no significan nada:**

| Columna | Qué es en castellano |
|---------|----------------------|
| `Apropiación Vigente` | Lo que el presupuesto **autoriza** gastar. El cupo |
| `Compromisos` | Lo que ya se **firmó** con un tercero. El contrato existe |
| `Obligaciones` | Lo que ya se **recibió** y hay que pagar. La factura llegó |
| `Pagos` | La plata que efectivamente **salió** |

La cadena es **cupo → contrato → factura → plata**. Cada eslabón es menor o igual al anterior, y ahí
está toda la historia de hoy: en cuánto se encoge la plata al pasar de un eslabón al siguiente.

**Las otras columnas que se usan:**

| Columna | Qué es |
|---------|--------|
| `Nombre Sector` | El sector del gasto: EDUCACION, TRANSPORTE, DEFENSA Y POLICIA... (32 en total) |
| `Nombre Nivel Uno Rubro` | El tipo de gasto: FUNCIONAMIENTO, INVERSION, SERVICIO DE LA DEUDA PÚBLICA |

Hay cinco niveles de rubro anidados (nivel uno a nivel cinco). Hoy solo se usa el nivel uno, y eso es
**ignorar información a propósito porque la historia no la necesita**. Decirlo en voz alta es parte
del oficio: un análisis que no declara qué dejó por fuera no se puede auditar.

In [ ]:
import pandas as pd

# Este notebook vive en clase09/demo/. Cada `..` sube un nivel de carpeta: dos saltos
# llegan a la raíz del repositorio, y de ahí se entra a datasets/.
df = pd.read_csv("../datos/EJECUCION_PRESUPUESTAL.csv")

# .shape devuelve la tupla (filas, columnas). Debe dar (3645, 21).
print(df.shape)

In [ ]:
# Los 21 nombres de columna, exactamente como están escritos en el archivo.
for columna in df.columns:
    print(columna)

### El primer vistazo, que hoy tampoco se salta

La historia que se va a contar sale de estas columnas, así que primero hay que saber qué son. Son
las mismas cuatro preguntas de la clase 2 y la clase 3, con las mismas funciones. **Una narrativa
construida sobre una columna mal entendida es una narrativa falsa que suena bien**, y ese es el
riesgo propio de hoy: el storytelling hace creíble cualquier cosa, incluido un error.

| Pregunta | Función |
|----------|---------|
| ¿Cuántas filas y columnas hay? | `df.shape` |
| ¿De qué tipo es cada columna y cuántos no-nulos tiene? | `df.info()` |
| ¿Los números están guardados como números? | `df.dtypes` |
| ¿Dónde hay nulos? | `df.isna().sum()` |
| ¿Qué tan grandes son las cifras con las que se va a contar la historia? | `df.describe()` |
| ¿Cuántas categorías distintas hay, y cómo se reparten? | `df[col].nunique()`, `df[col].value_counts()` |

In [ ]:
# ¿De qué tipo es cada columna y cuántos valores no nulos tiene cada una?
df.info()

In [ ]:
# ¿Los números están guardados como números? Si una columna de plata saliera como object
# (texto), sumarla concatenaría cadenas en vez de sumar pesos.
print(df.dtypes)

In [ ]:
# ¿Dónde hay nulos y cuántos? Solo se listan las columnas que tienen al menos uno.
nulos = df.isna().sum()
print(nulos[nulos > 0] if (nulos > 0).any() else "Ninguna columna tiene nulos.")

In [ ]:
# ¿De qué tamaño son las cifras de la historia? .T pone una columna por fila, que se lee mejor.
df[["Apropiación Vigente", "Compromisos", "Obligaciones", "Pagos"]].describe().T

In [ ]:
# ¿Cuántas categorías distintas hay en las columnas que van a estructurar el relato?
print("Sectores distintos:", df["Nombre Sector"].nunique())
print("Entidades distintas:", df["Nombre Entidad"].nunique())
print()

# value_counts: cuántas filas aporta cada tipo de gasto.
print(df["Nombre Nivel Uno Rubro"].value_counts())

**Lo que este vistazo ya deja dicho.** Las cuatro columnas de plata son numéricas, así que se
pueden sumar sin convertir nada. Hay 32 sectores y 221 entidades: el sector es una unidad que cabe en
una lámina, la entidad no. Y `Nombre Nivel Uno Rubro` tiene tres valores —FUNCIONAMIENTO, INVERSION,
SERVICIO DE LA DEUDA PÚBLICA— repartidos de forma muy despareja. Ese reparto desparejo es materia
prima narrativa: una historia necesita un contraste, y `value_counts()` es donde suele aparecer el
primero.

### 1.1 El error provocado: el nombre de columna que uno cree que existe

Mire la lista de arriba con cuidado. Hay dos nombres sucios:

- `Nombre Niel Dos Rubro` — le falta la **v** de "Nivel".
- `Recursos´Presupuestales` — lleva una tilde suelta donde debería ir un espacio.

**El concepto: el nombre de la columna en su código tiene que ser exactamente el del archivo, con
sus errores incluidos.** pandas no adivina, no corrige y no se parece. La celda de abajo provoca el
error a propósito para que lo reconozca cuando le salga solo.

In [ ]:
try:
    # El nombre correcto en castellano. Y por eso mismo, el que NO está en el archivo.
    df["Nombre Nivel Dos Rubro"]
except KeyError as error:
    print("Escribiendo el nombre bien ->", type(error).__name__)
    print("  la columna que pedí:", error)

print()
print("El archivo dice:", [c for c in df.columns if "Dos Rubro" in c])
print()
print("Un KeyError sobre un DataFrame casi siempre es un nombre mal escrito, no una columna que")
print("falta. La forma rápida de salir: buscar el nombre real dentro de df.columns, como arriba.")

> **Para entender qué está pasando · por qué no se corrige el archivo fuente**
>
> La tentación es abrir el CSV y arreglar el typo. No se hace, por dos razones.
>
> **La primera es de reproducibilidad.** El archivo se descargó de datos.gov.co. Si usted lo edita a
> mano, su notebook deja de correr sobre el dato público y empieza a correr sobre su versión privada
> de él. El día que alguien vuelva a descargar el archivo, su código revienta y nadie va a saber por
> qué.
>
> **La segunda es de honestidad.** El dato tiene los defectos que tiene, y eso es información sobre
> la fuente. Se documenta y se usa tal cual. Si el nombre estorba mucho, se renombra **en el código**
> con `df.rename(columns={...})` (clase 3), que es una transformación visible y reversible, no una
> edición silenciosa del origen.
>
> Esta distinción vuelve en la sustentación: "¿de dónde salió ese número?" tiene una respuesta buena
> solo si la cadena desde el archivo original hasta la cifra está escrita en algún lado.

**Pregunta de interpretación 1.** El archivo público trae dos nombres de columna con defecto. Si
usted necesitara ese nombre limpio para el resto de su análisis, ¿cuál de estas tres salidas elegiría
y por qué: editar el CSV, renombrar con `df.rename` en el código, o dejarlo tal cual y escribirlo mal
a propósito cada vez? Piense la respuesta desde quien va a auditar su trabajo, no desde su comodidad.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Renombrar en el código**, y dejar constancia de que se renombró.

Editar el CSV rompe la reproducibilidad: su notebook deja de correr sobre el dato público y pasa a
correr sobre su copia privada. El día que alguien vuelva a descargar el archivo, su código revienta y
nadie va a entender por qué.

Escribirlo mal cada vez funciona, pero convierte el typo en una trampa repartida por cincuenta celdas:
basta que una sola vez lo escriba bien para que reviente, y el error va a aparecer lejos de su causa.

`df.rename(columns={...})` es una transformación **visible y reversible**: está en el código, se lee,
se audita y se deshace. Es la misma lógica de toda la clase 3: **el dato sucio no se esconde, se
documenta y se transforma a la vista**.

Y hay una lectura extra: un archivo público con un nombre de columna sin la "v" es información sobre
la fuente. Dice que nadie revisó el esquema antes de publicarlo, y eso es un dato que vale la pena
mencionar cuando alguien pregunte qué tan confiable es el origen.

</details>

---

## 2. Las cuatro cifras del país

**El concepto: `sum()` sobre varias columnas devuelve una Series con un total por columna.** No es
un número: es cuatro números con nombre. Se usa `df[lista_de_columnas].sum()`, donde la lista va
entre corchetes dobles porque seleccionar varias columnas devuelve un DataFrame (clase 2, sección 3).

**El segundo concepto, y es el que separa un análisis de una slide: la magnitud tiene que ser
legible.** Los valores están en pesos, sin separador, con doce y trece dígitos. Nadie lee trece
dígitos. Dividir entre `1e12` los convierte en **billones de pesos** (un billón = un millón de
millones). Esa división no es cosmética: es la diferencia entre una slide que se entiende y una que
no.

In [ ]:
# Primero, el error que NO lanza ninguna excepción: la cifra cruda.
print("Compromisos, en pesos:  ", df["Compromisos"].sum())
print("Compromisos, en billones:", round(df["Compromisos"].sum() / 1e12, 2))
print()
print("Las dos líneas dicen lo mismo. Solo una de las dos se puede decir en voz alta.")

### 2.1 Las cuatro cifras del país, en billones

`df[columnas_dinero]` selecciona las cuatro columnas de golpe y devuelve un DataFrame; `.sum()` sobre
ese DataFrame devuelve una **Series con un total por columna**, no un número. La división entre
`BILLON` se aplica a los cuatro totales de una vez, porque dividir una Series entre un número divide
cada elemento.

In [ ]:
BILLON = 1e12

columnas_dinero = ["Apropiación Vigente", "Compromisos", "Obligaciones", "Pagos"]

totales = df[columnas_dinero].sum() / BILLON

print(totales.round(2))

---

## 3. La brecha, y el número que sí es un titular

Aquí no hay pandas nuevo. Es una resta y una división. Lo que sigue es narrativo, no técnico.

**El concepto: una resta y un porcentaje no comunican lo mismo.**

- La **resta** (compromisos menos pagos) dice *cuánta* plata falta. Sirve para dimensionar.
- El **porcentaje** (pagos sobre compromisos) dice *qué proporción* se cumplió. Sirve para juzgar.

Las dos cifras salen del mismo par de números y las dos son verdaderas. Cuál va en el título depende
de a quién le esté hablando, y eso se decide en la sección 8, no aquí.

### 3.1 La brecha y el porcentaje de ejecución

El porcentaje se calcula sobre los totales ya convertidos a billones: da exactamente lo mismo,
porque el factor se cancela en el cociente. Y va multiplicado por 100, porque 0,274 no es un
porcentaje: es una proporción, y decir "0,27% de ejecución" en una slide es un error de dos órdenes
de magnitud que nadie corrige en voz alta.

In [ ]:
compromisos = df["Compromisos"].sum() / BILLON
pagos = df["Pagos"].sum() / BILLON

brecha = compromisos - pagos

ejecucion = pagos / compromisos * 100

print(f"Compromisos: {compromisos:,.2f} billones")
print(f"Pagos:       {pagos:,.2f} billones")
print(f"Brecha:      {brecha:,.2f} billones")
print(f"Ejecución:   {ejecucion:,.1f}%")

**Pregunta de interpretación 2.** Con los cuatro números de `totales` a la vista: ¿en cuál eslabón
de la cadena cupo → contrato → factura → plata se pierde más? ¿Y cuál de las dos cifras (la brecha en
billones o el porcentaje de ejecución) pondría usted en el título de una slide dirigida a un
ministro?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Dónde se pierde más:** entre el cupo y el contrato. De 546,98 billones autorizados solo se firmaron
92,34: ahí se cae el 83% de la plata. La caída de contrato a pago (92,34 a 25,32) es la segunda, y es
la que se cuenta hoy porque es la que tiene consecuencias visibles: hay contratos firmados con gente
que está esperando que le paguen.

Ojo con la trampa: la primera caída es enorme pero puede ser normal según el corte del archivo (a
mitad de año todavía no se ha comprometido todo el cupo). La segunda es más difícil de explicar así.
**Distinguir la caída que el calendario explica de la que no, es análisis; presentar las dos como
escándalo, es opinión.**

**Qué cifra al título:** el porcentaje. "Se ejecutó el 27%" obliga a un juicio inmediato, y no exige
que la audiencia sepa cuánto es mucho en billones. Los 67 billones van en el cuerpo de la slide,
sosteniendo el título.

</details>

**Pregunta de interpretación 3.** Un periodista lee su cifra y titula *"El Gobierno dejó de pagar
67 billones de pesos"*. ¿Qué parte de esa frase sostiene el dato y qué parte no? Escriba la versión
que usted sí defendería en una sustentación.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Lo que el dato sostiene:** que la diferencia entre lo comprometido y lo pagado, al corte de este
archivo, es de 67,02 billones de pesos. Eso es una resta y se puede recalcular delante de quien
pregunte.

**Lo que no sostiene:** el verbo *"dejó de pagar"*. Implica que el plazo venció y que el pago era
exigible, y eso no está en el CSV. De hecho la columna `Obligaciones` (25,51 billones) dice que solo
una parte de esos compromisos ya tiene factura recibida: lo demás son contratos firmados cuyo pago
todavía no se ha causado. Confundir *comprometido* con *vencido* es exactamente el error que el
glosario de la sección 1 existe para evitar.

**Una versión defendible:** "De los 92 billones que el Estado comprometió, se han pagado 25. La
brecha es de 67 billones al corte de este archivo." Descriptiva, con la fecha del corte al lado, y
sin adjudicar intención ni incumplimiento.

Regla que se lleva: **el verbo es donde se cuelan las afirmaciones que el dato no aguanta.** "Hay una
brecha" es descripción; "dejó de pagar", "incumplió" o "se robaron" son acusaciones, y ninguna sale
de una resta.

</details>

> **Para entender qué está pasando · esto no es maquillar los datos**
>
> Es la objeción que aparece siempre en esta clase, y es sana. La respuesta tiene un límite claro.
>
> **Maquillar** es cambiar el número, esconder el que no conviene, o elegir el eje del gráfico para
> que una diferencia de dos puntos parezca un abismo (clase 8). **Contar** es elegir el orden en que
> se dicen números verdaderos, y elegir cuál va primero.
>
> La prueba práctica es una sola: **¿puede mostrar el cálculo cuando se lo pidan?** Por eso este
> notebook existe y por eso la regla de la clase es que una cifra que no puede recalcular no va en
> una slide. La historia tiene un cuaderno detrás, y ese cuaderno es lo que la separa del discurso.

---

## 4. ¿Dónde está la brecha?

**El concepto: un total es un titular; una historia necesita un nombre propio.** 67 billones es una
cifra que la audiencia no puede hacer nada con ella. "Educación concentra un tercio" ya señala a
alguien.

`groupby` se vio en la clase 4: separar las filas en montones según el valor de una columna y
calcular algo en cada montón. La analogía de los M&Ms sigue valiendo — separar por color y pesar cada
montón. Después `sort_values` ordena y `head` corta.

**Lo importante de esta sección no es la sintaxis: es que ordenar no cambia ningún dato y aun así
cambia la historia.** La tabla desordenada y la ordenada contienen exactamente la misma información,
y solo una de las dos se puede contar.

### 4.1 Los cinco sectores con más brecha

Tres pasos encadenados: agrupar por sector y sumar, agregar la columna `Brecha` con la resta, ordenar
de mayor a menor y cortar en cinco. `ascending=False` es lo que pone arriba lo grande, y `head(5)` es
una decisión narrativa, no técnica: cinco filas se leen en voz alta, treinta y dos no.

In [ ]:
por_sector = df.groupby("Nombre Sector")[["Compromisos", "Pagos"]].sum() / BILLON
por_sector["Brecha"] = por_sector["Compromisos"] - por_sector["Pagos"]

top_brecha = por_sector.sort_values("Brecha", ascending=False).head(5)

print(top_brecha.round(2))

**Pregunta de interpretación 4.** Mire la primera fila de su tabla. ¿Qué fracción de los 67,02
billones concentra ese solo sector? ¿Y por qué esa fracción es mejor material narrativo que la cifra
absoluta?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

Educación concentra 20,29 de los 67,02 billones: casi un tercio, con 20,29 / 67,02 = 30,3%.

La fracción es mejor material porque **es autosuficiente**. "20,29 billones" obliga a la audiencia a
recordar el total y hacer la división mentalmente mientras usted sigue hablando; nadie lo hace, así
que la cifra se pierde. "Un tercio de la brecha está en un solo sector" no necesita nada más y se
recuerda al día siguiente.

Regla general que sirve para cualquier dataset: **cuando una cifra absoluta necesita otra cifra para
entenderse, presente la relación entre las dos, no las dos.**

</details>

**Pregunta de interpretación 5.** `sort_values` no cambió un solo dato: la tabla ordenada y la
desordenada contienen exactamente la misma información. Entonces, ¿por qué solo una de las dos se
puede contar? Y una segunda, más incómoda: cortar en cinco deja treinta y dos sectores por fuera.
¿En qué caso ese corte dejaría de ser una decisión narrativa y pasaría a ser un sesgo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Por qué solo la ordenada se cuenta.** Porque la atención de quien escucha es un recurso escaso y se
gasta en el orden en que usted la gaste. Una tabla alfabética obliga a la audiencia a hacer el trabajo
de buscar el máximo; una tabla ordenada ya lo hizo. Ordenar no agrega información: **convierte
información en jerarquía**, y una jerarquía es lo primero que hace falta para que haya historia.

**Cuándo el corte es sesgo.** Cuando lo que queda fuera contradice lo que queda dentro y usted no lo
dice. Aquí el top 5 es honesto porque la cola es pequeña y la conclusión ("un tercio de la brecha está
en un solo sector") no cambia al mirar los 32. Sería sesgo si, por ejemplo, el sexto sector tuviera
una brecha casi igual a la del primero y usted lo omitiera para que el primero pareciera único, o si
cortara en cinco justo donde empieza a aparecer un sector que le incomoda.

La prueba práctica: **¿la frase que voy a decir sobreviviría si mostrara la tabla completa?** Si sí,
el corte es edición. Si no, el corte es el argumento, y eso ya es otra cosa.

</details>

---

## 5. El recorte que cambia la conclusión

**El concepto: una cifra sin su recorte no significa nada.** El mismo dato, partido por otra columna,
puede sostener una historia distinta o desmentir la que usted iba a contar. Y una cifra que circula
sin decir de qué recorte salió es una cifra que nadie puede verificar.

Esta sección existe por un caso real de este mismo curso: **materiales anteriores contaban este caso
con una brecha de 28,9 billones**, y la celda de la sección 3 dio 67,02. Los dos números son
correctos y en un momento va a quedar claro por qué. No lo adelante: sáquelo usted.

### 5.1 La brecha por tipo de gasto

Es el mismo patrón de la sección 4 cambiando una sola cosa: la columna por la que se agrupa. En vez
de `Nombre Sector` (32 valores) se usa `Nombre Nivel Uno Rubro`, que tiene tres:
`FUNCIONAMIENTO`, `INVERSION` y `SERVICIO DE LA DEUDA PÚBLICA`.

`tabla.loc['fila', 'columna']` lee **una celda concreta** de un DataFrame por su nombre de fila y de
columna. Sirve para sacar un solo número de una tabla y meterlo en un título.

**Antes de ejecutar, apueste.** Mire las tres categorías y escriba mentalmente cuál cree que ejecuta
mejor y cuál peor. La celda de abajo va a estar de acuerdo con usted o no, y las dos cosas enseñan.

In [ ]:
por_tipo = df.groupby("Nombre Nivel Uno Rubro")[["Compromisos", "Pagos"]].sum() / BILLON
por_tipo["Brecha"] = por_tipo["Compromisos"] - por_tipo["Pagos"]
por_tipo["Ejecución %"] = por_tipo["Pagos"] / por_tipo["Compromisos"] * 100

brecha_inversion = por_tipo.loc["INVERSION", "Brecha"]

print(por_tipo.round(2))

**Ahí están los 28,9 billones.** La cifra con la que se contaba este caso el semestre pasado no era
otra cifra ni un error: era **la brecha del gasto de inversión** (28,85), no la brecha total (67,02).
Es un recorte del mismo dato, y los dos números son correctos al mismo tiempo.

Y la historia acaba de cambiar de dueño:

| Tipo de gasto | Comprometido | Pagado | Brecha | Ejecución |
|---------------|--------------|--------|--------|-----------|
| Servicio de la deuda | 8,42 | 8,21 | 0,21 | **97,5%** |
| Funcionamiento | 54,01 | 16,06 | 37,95 | 29,7% |
| Inversión | 29,91 | 1,06 | **28,85** | **3,5%** |

La deuda se paga casi entera. La inversión —que es la plata de las carreteras, los colegios y los
acueductos— ejecuta al 3,5%. **Ese contraste es la historia de hoy, y no aparecía en ninguna de las
cifras totales de las secciones anteriores.**

**Pregunta de interpretación 6.** Usted acaba de calcular dos brechas verdaderas sobre el mismo
archivo: 67,02 y 28,85 billones. ¿Qué habría que decir al lado de cada una para que la frase sea
completa? Y la pregunta que importa: si alguien le muestra en una reunión la cifra de 28,85 sin decir
de qué recorte salió, ¿qué le pregunta usted?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Lo que va al lado de cada una:** el recorte. "67,02 billones de brecha **entre todo lo comprometido
y todo lo pagado**" y "28,85 billones de brecha **en el gasto de inversión**". Sin esa coletilla, las
dos cifras son intercambiables en la cabeza de quien escucha, y una de las dos es tres veces la otra.

**Qué se le pregunta a alguien que muestra un número suelto:** *¿sobre qué subconjunto está calculado
eso?* Es la pregunta más barata y más productiva de una reunión de datos. Le sigue *¿en qué fecha
corta el archivo?*

**Por qué esto importa más que la aritmética:** una cifra sin su recorte no es una cifra
aproximadamente correcta, es una cifra **sin significado**, porque hay infinitos subconjuntos del
mismo dataset que producen infinitos números y todos son verdaderos. El recorte es lo que convierte
un número en una afirmación.

Y hay una lección de oficio: este curso arrastraba la contradicción entre 28,9 y 67,02 desde el
semestre pasado, y nadie la había notado porque las dos cifras venían de un cálculo correcto. **Los
errores más difíciles de encontrar no son los cálculos malos: son los cálculos buenos sobre el
recorte equivocado.**

</details>

> **Para entender qué está pasando · dónde está la línea entre analítica y opinión**
>
> Con la tabla de arriba a la vista, hay una frase que casi todo el mundo quiere decir: *"primero se
> paga lo que tiene consecuencias jurídicas y de último lo que tiene consecuencias sociales"*.
>
> Esa frase es una **interpretación causal sobre una intención**, y el dato no la sostiene. Lo que el
> dato sostiene es la descripción: la deuda ejecuta al 97,5% y la inversión al 3,5%. El *porqué*
> puede ser prioridad política, puede ser que los pagos de deuda son automáticos y los de inversión
> requieren interventoría y actas, puede ser el corte del archivo. Son tres explicaciones distintas y
> ninguna está en el CSV.
>
> **La regla que se lleva:** distinguir lo que el dato aguanta de lo que uno quisiera que dijera es
> exactamente la línea entre analítica y opinión. Y cruzarla en una sustentación es la forma más
> rápida de perder la sala, porque la primera pregunta va a ser *"¿de dónde saca usted eso?"* y no va
> a tener respuesta.
>
> La misma advertencia aplica a la palabra "robo". El dato soporta *"hay una brecha"*. No soporta
> *"se robaron la plata"*. Puede ser corrupción, puede ser lentitud administrativa, puede ser el
> corte del archivo a mitad de año.

**Pregunta de interpretación 7.** Escriba **dos frases** sobre la tabla de los tres tipos de gasto:
una que el dato sostenga y una que no, lo más parecidas entre sí que pueda. Después explique en una
línea qué palabra exacta es la que cruza la frontera.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Un par posible:**

- Sostenida: *"El servicio de la deuda ejecuta al 97,5% y la inversión al 3,5%."*
- No sostenida: *"El Estado prioriza pagarle a los bancos antes que construir colegios."*

**La palabra que cruza la frontera es `prioriza`**, porque atribuye una intención. El CSV tiene
montos, no decisiones. Hay al menos tres explicaciones compatibles con esos mismos números: prioridad
política, diferencia de trámite (los pagos de deuda son automáticos y los de inversión requieren
interventoría, actas y recibo a satisfacción), o el corte del archivo a mitad de año, que castiga
más a la inversión porque su ejecución se concentra al final. **Ninguna de las tres está en el
archivo, y por eso ninguna se afirma.**

Cómo se dice bien cuando uno sí quiere ir ahí: *"La inversión ejecuta al 3,5%. Una hipótesis que
habría que verificar con los tiempos de trámite es que..."*. Nombrarla como hipótesis no le quita
fuerza: le quita el flanco por el que se pierde la sala.

**El costo de cruzarla en una sustentación** es concreto: la primera pregunta va a ser *"¿de dónde
saca usted eso?"*, y no va a haber respuesta. A partir de ahí todo lo demás que diga, incluido lo que
sí estaba bien calculado, se escucha con desconfianza.

</details>

---

# Segunda mitad — la historia

**Hasta aquí llegó el código. Cierre mentalmente el portátil.**

> Ya tenemos el número. El número no es la historia.

Lo que sigue son cuatro celdas markdown: contexto, hallazgo, implicación y acción. Cada una arranca
con **la versión mala**, para que se vea qué se está corrigiendo. Escribir mal a propósito y
corregirse es la mitad del valor de este tramo.

**Las cifras disponibles, todas calculadas arriba:**

| Concepto | Valor |
|----------|-------|
| Apropiación vigente | 546,98 billones |
| Compromisos | 92,34 billones |
| Obligaciones | 25,51 billones |
| Pagos | 25,32 billones |
| Brecha compromisos menos pagos | 67,02 billones |
| Ejecución (pagos / compromisos) | 27,4% |
| Sector con mayor brecha | Educación, 20,29 billones |
| Brecha del gasto de inversión | 28,85 billones |
| Ejecución de la inversión | 3,5% |
| Ejecución del funcionamiento | 29,7% |
| Ejecución del servicio de la deuda | 97,5% |

**El marco, que es lo que se replica en el reto:**

| Slide | Pregunta que responde | Error típico |
|-------|----------------------|--------------|
| 1. Contexto | ¿De qué estamos hablando y por qué ahora? | Convertirlo en la agenda de la presentación |
| 2. Hallazgo | ¿Qué encontramos, en una frase con cifra? | Meter tres hallazgos |
| 3. Implicación | ¿Y eso qué significa para quien decide? | Repetir el hallazgo con otras palabras |
| 4. Acción | ¿Qué hay que hacer, quién y cuándo? | Terminar con "gracias, ¿preguntas?" |

## 6.1 Slide 1 · Contexto

> **Versión débil:** "Análisis de la ejecución presupuestal nacional"
>
> Qué está mal: anuncia un tema. El que escucha todavía no sabe por qué debería quedarse en la sala.

### Colombia autorizó 547 billones de gasto este año. La pregunta es cuánto llegó a alguien.

- El Presupuesto General de la Nación autorizó **546,98 billones** de pesos.
- Ese número es el cupo, no la plata que salió.
- Entre el cupo y la plata hay tres eslabones: contrato, factura y pago.

**Por qué esta versión funciona:** instala una pregunta en vez de anunciar un tema, y la pregunta es
la que el resto del deck va a responder. El contexto no es "de qué voy a hablar": es **por qué esto
importa ahora**.

## 6.2 Slide 2 · Hallazgo

> **Versión débil:** "Resultados del análisis por sector"
>
> Qué está mal: es el nombre de una sección de un informe, no un hallazgo. Y promete varios
> hallazgos, de los cuales la audiencia se va a llevar cero.

### Se comprometieron 92 billones. Se pagaron 25.

- **92,34 billones** en contratos firmados.
- **25,32 billones** efectivamente pagados.
- **27,4%** de ejecución real.

**Por qué esta versión funciona:** dos cifras, una resta que el auditorio hace solo, y un contraste
que no necesita explicación.

**Cuántas cifras caben aquí:** tres es el techo, y con dos suele bastar. Cada cifra adicional divide
la atención.

**Cuántos gráficos:** uno. Un gráfico consume unos 45 segundos de atención real si lo explica bien, y
no sirve de nada si no lo explica.

## 6.3 Slide 3 · Implicación

> **Versión débil:** "Existe una brecha de 67 billones entre compromisos y pagos"
>
> Qué está mal: eso es el hallazgo otra vez, con otras palabras. **Es la trampa más común de las
> cuatro slides**, y en la que más gente se atasca, incluido el profesor. Si la 3 dice lo mismo que
> la 2, sobra una.
>
> El método para salir: preguntarse **"¿y eso qué?"** tres veces seguidas, hasta que la frase deje de
> ser un dato y empiece a ser una consecuencia para alguien.

### La historia no es contable. Es de ejecución.

- Si el tablero de control solo mira compromisos, el país se ve **tres veces mejor** de lo que está.
- La deuda ejecuta al **97,5%**; la inversión, al **3,5%**.
- Quien decide con la métrica de compromisos está decidiendo sobre plata que no se ha movido.

**Por qué esta versión funciona:** cambia lo que la audiencia creía al entrar. Antes creía que el
problema era de monto; sale sabiendo que es de métrica.

**La prueba mecánica del "¿y eso qué?":** si su slide 3 comparte la mitad de sus palabras con la
slide 2, casi seguro está repitiendo. El verificador del reto revisa exactamente eso.

**Pregunta de interpretación 8.** Esta es la sección donde se atasca todo el mundo, así que vale la
pena atascarse aquí y no en el reto. Escriba **una implicación distinta** a la del cuaderno, partiendo
del mismo hallazgo pero dirigida a otro decisor: no al que arma el tablero de control, sino a un
contratista que está evaluando si presentarse a una licitación del Estado. Después revise la suya con
la prueba del solapamiento: ¿comparte más de la mitad de sus palabras con el hallazgo?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Una implicación posible para ese decisor:** *"Firmar con el Estado y cobrarle son dos cosas
distintas, y hay que financiar el hueco entre las dos."* El mismo hallazgo (92 comprometidos, 25
pagados) deja de ser una cifra macro y se vuelve un problema de flujo de caja de quien escucha.

**Por qué funciona:** cambia una decisión concreta. Quien la oye va a mirar distinto la cláusula de
plazos de pago del pliego, o va a pedir un anticipo mayor. Eso es lo que significa "implicación":
**alguien tiene que poder decidir distinto por haberla oído**.

**Por qué "existe una brecha de 67 billones" no lo es:** repite el hallazgo con más palabras. Compare
el vocabulario de las dos frases: brecha, billones, compromisos, pagos. Si las palabras se repiten,
el pensamiento se repitió.

**El método, otra vez:** "¿y eso qué?" tres veces. *Hay una brecha* → ¿y eso qué? → *hay contratos
firmados sin pagar* → ¿y eso qué? → *alguien está financiando esa plata mientras tanto* → ¿y eso qué?
→ *si usted contrata con el Estado, ese alguien es usted*. Ahí paró: la cuarta frase ya no es un dato,
es una consecuencia con dueño.

</details>

## 6.4 Slide 4 · Acción

> **Versión débil:** "Se recomienda hacer seguimiento a la ejecución"
>
> Qué está mal: no dice quién, ni qué métrica, ni por dónde empezar. Es una forma educada de no
> concluir.

### Reportar pagos, no compromisos, en el tablero mensual. Empezando por Educación.

- **Qué:** cambiar la métrica principal del tablero de compromisos a pagos.
- **Quién:** Planeación Nacional, con los enlaces de cada sector.
- **Por dónde:** Educación, que concentra 20 de los 67 billones.
- **Cuándo:** próximo corte mensual.

**Por qué esta versión funciona:** alguien puede hacer algo distinto el lunes por haber escuchado
esto. Ese es el estándar, y es más exigente de lo que parece.

**Criterio para revisar la suya:** la acción nombra un responsable, una métrica y un punto de
arranque. Si le falta alguno de los tres, todavía no es una acción.

**Pregunta de interpretación 9.** Aquí van tres cierres reales del tipo que aparece en las
sustentaciones. Diga cuál de los tres criterios (qué, quién, por dónde empezar) le falta a cada uno y
arregle el peor de los tres.

1. *"Se recomienda profundizar el análisis en trabajos futuros."*
2. *"Planeación Nacional debería mejorar la ejecución presupuestal."*
3. *"Cambiar la métrica del tablero a pagos."*

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

1. **Le faltan los tres.** No dice qué se profundiza, ni quién lo hace, ni por dónde. Es el peor de
   los tres y además es una forma educada de decir que no hubo conclusión. Arreglado:
   *"Planeación Nacional debe reportar pagos, no compromisos, en el tablero mensual, empezando por
   Educación, que concentra 20 de los 67 billones."*
2. **Tiene quién, le falta qué y por dónde.** "Mejorar la ejecución" no es una acción: es el deseo de
   que el problema no exista. Nadie sabe qué hacer el lunes con esa frase.
3. **Tiene qué, le faltan quién y por dónde.** Es la más cercana a servir, y por eso es la más
   peligrosa: suena concreta y no le asigna la tarea a nadie. Las recomendaciones sin dueño son las
   que aparecen idénticas en el informe del año siguiente.

**El estándar completo:** alguien tiene que poder hacer algo distinto el lunes por haber escuchado
esto. Si la frase no sobrevive a la pregunta *"¿quién, exactamente, y con qué empieza?"*, todavía no
es una acción.

</details>

---

## 7. La prueba de los títulos

**El concepto: el título de la slide es el mensaje, no la etiqueta del contenido.** Ningún periódico
titula "Cifras de accidentalidad vial en el primer trimestre"; titula "Las motos causan seis de cada
diez muertes en vía". El título ya dijo qué pensar y el cuerpo solo lo sostiene.

**La prueba, que toma 20 segundos:** lea los cuatro títulos seguidos, sin el cuerpo.

1. Colombia autorizó 547 billones de gasto este año. La pregunta es cuánto llegó a alguien.
2. Se comprometieron 92 billones. Se pagaron 25.
3. La historia no es contable. Es de ejecución.
4. Reportar pagos, no compromisos, en el tablero mensual. Empezando por Educación.

Eso ya es la historia completa. **Si sus títulos leídos en fila suenan a índice** ("Introducción",
"Metodología", "Resultados", "Conclusiones")**, el deck está mal armado aunque el contenido sea
correcto.**

| Título descriptivo (débil) | Título con mensaje (fuerte) |
|----------------------------|------------------------------|
| "Compromisos y pagos por sector" | "Educación concentra un tercio de la brecha" |
| "Distribución del gasto" | "Nueve de cada diez pesos comprometidos siguen sin pagarse" |
| "Análisis de correlación" | "Más presupuesto no está produciendo más ejecución" |

**Pregunta de interpretación 10.** Vaya al dataset de su equipo, el del proyecto, y escriba **dos
títulos** para el mismo gráfico que ya tenga hecho: el descriptivo que le saldría por defecto y el que
afirma un hallazgo con su cifra. Después responda: ¿qué tuvo que saber sobre sus datos para poder
escribir el segundo, que no necesitaba para escribir el primero?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

La respuesta depende de su dataset, pero lo que se busca es que note esto: **el título descriptivo se
puede escribir sin haber mirado el resultado.** "Distribución del consumo por estrato" se escribe
antes de correr la celda. El título que afirma exige haber leído el gráfico, haber decidido cuál de
las cosas que muestra es la importante y haber sacado la cifra que la sostiene.

Por eso el título descriptivo es tan común: es el que sale cuando uno **todavía no decidió qué está
contando**. Y por eso cambiarlo no es un retoque de redacción: obliga a hacer el análisis que faltaba.

La forma rápida de detectarlo en su propio deck: si puede intercambiar el título entre dos gráficos
distintos y ninguno de los dos queda mal, los dos títulos son etiquetas.

</details>

---

## 8. La misma verdad, tres audiencias

**El concepto: el hallazgo no cambia, el encuadre sí.** No es mentir tres veces: son tres niveles de
detalle sobre un solo hecho. La analogía es la receta médica: el mismo diagnóstico se cuenta distinto
al paciente, al familiar y al colega médico.

| Audiencia | Qué le importa | La misma verdad |
|-----------|----------------|-----------------|
| Ejecutivo (ministro, rector, gerente general) | La decisión y el riesgo | "Estamos comprometiendo tres veces más de lo que pagamos. Si el ritmo sigue, cerramos el año con la mayor parte de la ejecución sin materializar." |
| Gerente de área (jefe de planeación) | Dónde actuar y con qué recursos | "Educación concentra 20 de los 67 billones de brecha. Ahí es donde primero hay que revisar el proceso de pago." |
| Técnico (equipo de datos) | Cómo se calculó y qué tan confiable es | "Sumé `Compromisos` y `Pagos` por sector sobre 3.645 registros, sin nulos. La brecha es la resta directa; no ajusté por vigencias futuras." |

**La regla que se lleva: una presentación, una audiencia principal.** Si intenta servirle a las tres
en el mismo deck, las tres se aburren en momentos distintos.

Y una nota sobre esta aula en concreto: aquí hay Ingeniería de Sistemas e Ingeniería Industrial
juntos. Un mismo hallazgo sobre consumo de agua se cuenta distinto si quien escucha optimiza una red
de distribución o si diseña el sistema que la monitorea.

### 8.1 Qué se puede revisar por máquina de un título, y qué no

En el reto, dentro de una hora, usted va a escribir cuatro títulos y el cuaderno los va a revisar.
Conviene saber desde ya **qué está haciendo esa revisión**, porque de lo contrario se lee como una
calificación, y no lo es.

**Lo que el cuaderno del reto sí puede revisar, porque son fallas de forma y son mecánicas:**

| Lo que detecta | Cómo |
|----------------|------|
| La plantilla sin llenar | El texto sigue siendo el ejemplo que venía escrito |
| El arranque de índice | Empieza por "Análisis de", "Resultados", "Introducción" |
| El hallazgo sin cifra | No hay ningún número en la frase del hallazgo |
| La implicación que repite el hallazgo | Solapamiento léxico: comparte demasiadas palabras con la slide anterior |
| La acción que no concluye | No aparece nada que se parezca a un responsable ni a un verbo de decisión |
| La cifra del título que no se reproduce | Se recalcula desde el CSV y se compara. Esta sí es una comprobación de verdad |

**Lo que no puede revisar, y nunca va a poder:** si el título es *bueno*. Si la implicación es la que
importa para esa audiencia. Si la acción es realista. Si la historia se sostiene leída en voz alta.
Eso exige criterio, y automatizarlo requeriría fingir que existe una respuesta correcta cuando no
existe.

> **La frase que hay que llevarse al reto: verde significa "tiene forma de título", no significa "es
> un buen título".** Un título que pasa los seis chequeos puede ser aburrido, irrelevante o dirigido
> a la audiencia equivocada, y el cuaderno lo va a dejar pasar sin decir nada. El juicio lo cierra
> usted en la autoevaluación del reto, con criterios explícitos, y lo cierra la sala en la
> sustentación del Momento 2.

Que la máquina revise lo mecánico y usted revise lo que exige criterio no es una limitación del
cuaderno: es el reparto correcto. Lo que se puede verificar sin criterio, se verifica; **lo que no,
se declara en voz alta en vez de disfrazarse de semáforo**.

**Pregunta de interpretación 11.** Escoja una audiencia **distinta** a las tres de la tabla: un
periodista, un concejal, un estudiante de primer semestre, un contratista que lleva ocho meses
esperando que le paguen. Escriba el **título de la slide de hallazgo** dirigido a esa persona, y
debajo, una línea sobre qué tuvo que cambiar respecto al título del cuaderno y por qué. Después
aplíquele usted mismo los seis chequeos de arriba: ¿los pasaría? ¿Y basta con que los pase?

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

**Un ejemplo, para el contratista:** *"De cada 100 pesos que el Estado firmó con alguien como usted,
salieron 27."*

**Qué cambió respecto al título del cuaderno** ("Se comprometieron 92 billones. Se pagaron 25."):
desaparecieron los billones. Un contratista no dimensiona 92 billones y no le sirven para nada; lo
que le sirve es la proporción, porque es la que puede aplicar a su propio contrato. **El hallazgo es
el mismo, la unidad cambió para que la audiencia pudiera hacer algo con él.**

Otro ejemplo, para un estudiante de primer semestre: *"El Estado firmó cuatro veces más contratos de
los que alcanzó a pagar."* Sin cifras grandes, con una relación que se sostiene sola.

**Sobre los seis chequeos:** el título del ejemplo los pasa —afirma, tiene cifra, no arranca como
índice, cabe en una línea—. Y aun así podría ser un mal título: si la audiencia real fuera el
ministro, hablarle de "alguien como usted" sería raro y el 27% le diría menos que la brecha absoluta.
**Pasar los chequeos es el piso, no el techo.** La pregunta que cierra siempre es la misma: ¿esta
persona, en concreto, puede hacer algo distinto después de oír esto?

</details>

---

## 9. Lo que siempre se pregunta

**¿Por qué cuatro slides y no las que hagan falta?**
Porque la restricción es la que obliga a decidir. Con quince slides uno no elige qué es importante;
con cuatro sí. En la sustentación del Momento 2 son 7 minutos de exposición con corte duro, y ahí la
restricción deja de ser pedagógica.

**¿Y si mi hallazgo no es sorprendente?**
Un hallazgo esperado también sirve si confirma con evidencia algo que antes era solo intuición. Lo
que no sirve es no tener hallazgo: "los datos muestran variabilidad" no es un hallazgo.

**¿Puedo poner la metodología?**
Sí, al final o en un anexo. Nunca al principio. Si la audiencia la necesita la va a pedir, y ese es
el mejor momento para mostrarla: lo hace ver preparado en vez de defensivo.

**¿Qué hago si la audiencia es mixta, técnica y no técnica?**
Escriba para la más alta en la cadena de decisión y tenga el respaldo técnico listo para las
preguntas.

**¿Puedo usar IA para escribir mis slides?**
Para redactar y pulir, sí, y ya se vio cómo en la clase 7. Para inventar la cifra o el hallazgo, no.
Si no puede reproducir el número en su notebook, no va en la slide.

**¿Este dataset sirve para el proyecto de mi equipo?**
No. Los CSV de las clases son material de enseñanza, elegidos por lo que permiten enseñar. El dataset
del proyecto lo consigue cada equipo, y tiene que poder decir de dónde salió y bajo qué condiciones
se puede usar.

**Pregunta de interpretación 12.** Última, y es la que conecta con el bloque 3. Piense en el dataset
de su equipo y responda tres cosas en tres líneas: **quién** es su audiencia principal (una sola),
**qué** puede hacer distinto esa persona después de oír su presentación, y **cuál** de las cifras que
ya calcularon en los momentos anteriores sostiene eso. Si alguna de las tres líneas le sale vacía, esa
es exactamente la que hay que resolver en los próximos sesenta minutos.

*Tu respuesta:*

<details>
<summary>Comparar con la respuesta esperada</summary>

No hay respuesta modelo: es su proyecto. Lo que sí hay es un diagnóstico según cuál de las tres se
haya quedado en blanco.

**Si le faltó la audiencia:** es el caso más común y el más fácil de arreglar. Elija la más alta en la
cadena de decisión de las que podrían escucharlo, y sostenga esa. Un deck para "todo el mundo" aburre
a los tres tipos de oyente en momentos distintos.

**Si le faltó qué puede hacer distinto:** el problema no es de redacción, es que todavía no hay
hallazgo. Vuelva al EDA del Momento 1 y busque el contraste: dos grupos que se comportan distinto, una
tendencia que se rompe, un valor que no debería estar ahí. Sin contraste no hay historia, y ninguna
cantidad de gráficos lo compensa.

**Si le faltó la cifra:** tiene una intuición, no un hallazgo. Es recuperable y es trabajo de pandas:
la cifra sale de partir el dato por la columna correcta. **Pero mientras no exista, la afirmación no
va en una slide**, porque en la sustentación se la van a pedir.

</details>

---

## Resumen

| Lo que hizo | Con qué |
|-------------|---------|
| Ver los nombres reales de las columnas | `for columna in df.columns` |
| Totalizar varias columnas a la vez | `df[lista].sum()` |
| Volver legible una magnitud | dividir entre `1e3`, `1e6` o `1e12` |
| Partir el total por una categórica | `df.groupby('col')[[...]].sum()` |
| Convertir una tabla en un titular | `sort_values('col', ascending=False).head(5)` |
| Leer una celda concreta | `tabla.loc['fila', 'columna']` |
| Contrastar dos recortes del mismo dato | dos `groupby` sobre columnas distintas |

**Las seis reglas que no se negocian:**

1. La primera frase de su presentación es su conclusión, no su metodología.
2. El título de la slide es el mensaje. Si describe, está desperdiciado.
3. Contexto → hallazgo → implicación → acción. Cuatro slides, en ese orden.
4. La implicación no es el hallazgo con otras palabras. Pregúntese "¿y eso qué?".
5. La acción nombra qué, quién y por dónde empezar.
6. Una cifra que no puede recalcular no va en una slide.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, está listo para el reto:

- [ ] Puedo explicar la cadena cupo → contrato → factura → plata y decir en qué eslabón se pierde más.
- [ ] Sé por qué 28,85 y 67,02 son las dos correctas, y qué hay que decir al lado de cada una.
- [ ] Puedo convertir un título descriptivo en uno que afirme, sin ayuda.
- [ ] Sé distinguir una implicación de un hallazgo repetido, y tengo un método para salir del atasco.
- [ ] Puedo señalar en este cuaderno la frase exacta que cruzaría la línea entre analítica y opinión.

**Ahora en el reto:** lo mismo, con el dataset de su equipo, y lo presenta en 5 minutos.